# Credit Card Fraud Detection Case Study


## Environment Setup
This notebook is designed to run in both local Docker and AWS EMR environments. It automatically detects the environment and uses the appropriate data path.

In [13]:
from pyspark.sql import SparkSession

try:
    spark
except NameError:
    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("credit-card-fraud-detection") \
        .getOrCreate()

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

Spark version: 4.1.1
Spark master: local[*]


In [14]:
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, LongType,
    TimestampType, DateType
)

In [3]:
TEAM_DIRECTORY = "team_15"
DATA_FILENAME = "credit_card_transactions.csv"

def detect_environment() -> str:
    master = spark.sparkContext.master.lower()
    return "local" if master.startswith("local") else "emr"

def get_data_path() -> str:
    env = detect_environment()
    
    if env == "emr":
        return f"s3://msbx5420-2026/teams/{TEAM_DIRECTORY}/{DATA_FILENAME}"
    
    # Local (Docker) path resolution
    candidates = [
        Path.cwd() / "data" / "raw" / DATA_FILENAME,
        Path.cwd().parent / "data" / "raw" / DATA_FILENAME,
        Path.cwd() / DATA_FILENAME
    ]
    
    for path in candidates:
        if path.exists():
            return str(path)
    
    raise FileNotFoundError(
        f"Could not find {DATA_FILENAME} in expected locations: {candidates}"
    )

ENV = detect_environment()
RAW_DATA_PATH = get_data_path()

print(f"Environment: {ENV}")
print(f"Data path: {RAW_DATA_PATH}")

Environment: local
Data path: /home/jovyan/work/data/raw/credit_card_transactions.csv


## Data Ingestion
The dataset is loaded using an explicit schema to ensure consistent typing across both local and EMR environments.

In [4]:
TRANSACTION_SCHEMA = StructType([
    StructField("Unnamed: 0", LongType(), True),
    StructField("trans_date_trans_time", TimestampType(), True),
    StructField("cc_num", StringType(), True),
    StructField("merchant", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amt", DoubleType(), True),
    StructField("first", StringType(), True),
    StructField("last", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
    StructField("city_pop", IntegerType(), True),
    StructField("job", StringType(), True),
    StructField("dob", DateType(), True),
    StructField("trans_num", StringType(), True),
    StructField("unix_time", LongType(), True),
    StructField("merch_lat", DoubleType(), True),
    StructField("merch_long", DoubleType(), True),
    StructField("is_fraud", IntegerType(), True),
    StructField("merch_zipcode", StringType(), True),
])

In [5]:
df = (
    spark.read
         .option("header", True)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .option("dateFormat", "yyyy-MM-dd")
         .schema(TRANSACTION_SCHEMA)
         .csv(RAW_DATA_PATH)
)


print(f"Initial Dataset Shape:\n\tRow count: {df.count():,}\n\tColumn count: {len(df.columns)}")

Initial Dataset Shape:
	Row count: 1,296,675
	Column count: 24


## Data Cleaning and Feature Engineering
The dataset is cleaned and enriched with time-based, customer-level, and transaction-level derived features to support exploratory analysis and downstream modeling.

In [6]:
# Transform Dataframe
df = (
      df.withColumn("merchant", F.trim(F.col("merchant")))
        .withColumn("category", F.trim(F.col("category")))
        .withColumn("first", F.trim(F.col("first")))
        .withColumn("last", F.trim(F.col("last")))
        .withColumn("city", F.trim(F.col("city")))
        .withColumn("state", F.upper(F.trim(F.col("state"))))
        .withColumn("zip", F.trim(F.col("zip")))
        .withColumn("merch_zipcode", F.trim(F.col("merch_zipcode")))
        .withColumn("event_date", F.to_date(F.col("trans_date_trans_time")))
        .withColumn("event_hour", F.hour(F.col("trans_date_trans_time")))
        .withColumn("event_month", F.month(F.col("trans_date_trans_time")))
        .withColumn("event_dayofweek", F.dayofweek(F.col("trans_date_trans_time")))
        .withColumn("event_weekofyear", F.weekofyear(F.col("trans_date_trans_time")))
        .withColumn("age",F.floor(F.datediff(F.current_date(), F.col("dob")) / F.lit(365.25)))
        .withColumn("amt_log", F.log1p(F.col("amt")))
)

print(f"Transformed Dataset:\n\tRow count: {df.count():,}\n\tColumn count: {len(df.columns)}")

Transformed Dataset:
	Row count: 1,296,675
	Column count: 31


## Exploratory Data Analysis
This section reviews a sample of the engineered dataset, evaluates missing values, and summarizes the class distribution before restricting the dataset for modeling.

In [7]:
df.select(
    "amt",
    "category",
    "gender",
    "state",
    "event_hour",
    "event_dayofweek",
    "age",
    "is_fraud"
).show(10, truncate=False)

+------+-------------+------+-----+----------+---------------+---+--------+
|amt   |category     |gender|state|event_hour|event_dayofweek|age|is_fraud|
+------+-------------+------+-----+----------+---------------+---+--------+
|4.97  |misc_net     |F     |NC   |0         |3              |38 |0       |
|107.23|grocery_pos  |F     |WA   |0         |3              |47 |0       |
|220.11|entertainment|M     |ID   |0         |3              |64 |0       |
|45.0  |gas_transport|M     |MT   |0         |3              |59 |0       |
|41.96 |misc_pos     |M     |VA   |0         |3              |40 |0       |
|94.63 |gas_transport|F     |PA   |0         |3              |64 |0       |
|44.54 |grocery_net  |F     |KS   |0         |3              |32 |0       |
|71.65 |gas_transport|M     |VA   |0         |3              |78 |0       |
|4.27  |misc_pos     |F     |PA   |0         |3              |85 |0       |
|198.39|grocery_pos  |F     |TN   |0         |3              |52 |0       |
+------+----

In [8]:
missing_counts_row = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

missing_summary = spark.createDataFrame(
    [Row(column=col, missing_count=cnt) for col, cnt in missing_counts_row.items() if cnt > 0]
).orderBy(F.desc("missing_count"))

print("Columns with missing values:")
missing_summary.show(truncate=False)

Columns with missing values:
+-------------+-------------+
|column       |missing_count|
+-------------+-------------+
|merch_zipcode|195973       |
+-------------+-------------+



In [9]:
full_total = df.count()

full_fraud_dist = (
    df.groupBy("is_fraud")
      .count()
      .withColumn("count_formatted", F.format_number("count", 0))
      .withColumn("percentage", F.round(F.col("count") / full_total * 100, 4))
      .orderBy("is_fraud")
)

print("Number and percentage of fraud/non-fraud transactions in the transformed dataset:")
full_fraud_dist.select("is_fraud", "count_formatted", "percentage").show(truncate=False)

Number and percentage of fraud/non-fraud transactions in the transformed dataset:
+--------+---------------+----------+
|is_fraud|count_formatted|percentage|
+--------+---------------+----------+
|0       |1,289,169      |99.4211   |
|1       |7,506          |0.5789    |
+--------+---------------+----------+



## Modeling Dataset Preparation
The modeling dataset is restricted to selected numeric and categorical predictors and rows with complete values.

In [10]:
numeric_features = [
    "amt",
    "city_pop",
    "event_hour",
    "event_dayofweek",
    "age",
]

categorical_features = [
    "category",
    "gender",
    "state",
]

label_col = "is_fraud"

In [11]:
model_cols = numeric_features + categorical_features + [label_col]

model_df = df.select(*model_cols).dropna()

total = model_df.count()

print(f"Modeling row count after dropna: {total:,}\n")

fraud_dist = (
    model_df.groupBy("is_fraud")
            .count()
            .withColumn("count_formatted", F.format_number("count", 0))
            .withColumn("percentage", F.round(F.col("count") / total * 100, 4))
            .orderBy("is_fraud")
)

print("Number and percentage of fraud/non-fraud transactions in the modeling dataset:")
fraud_dist.select("is_fraud", "count_formatted", "percentage").show(truncate=False)

Modeling row count after dropna: 1,296,675

Number and percentage of fraud/non-fraud transactions in the modeling dataset:
+--------+---------------+----------+
|is_fraud|count_formatted|percentage|
+--------+---------------+----------+
|0       |1,289,169      |99.4211   |
|1       |7,506          |0.5789    |
+--------+---------------+----------+

